# Dataset Relabeling

Before training, we need to merge similar classes and remove classes with too few samples:
- **bench** + **couch** → **chair** (all represent seating)
- **handbag** → **backpack** (similar bag types)
- **keyboard** → DELETED (only 3 samples)

This will improve model performance by increasing samples per class and removing problematic small classes.

In [ ]:
# Run the relabeling script
%run relabel_dataset.py

In [1]:
import os
import torch
from ultralytics import YOLO
import logging
logging.basicConfig(level=logging.INFO) # Setup logging
logging.info("Import library successful !")

INFO:root:Import library successful !


In [2]:
# Initialize the environment variable
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

# Dataset and Project Configuration
DATASET_ROOT = r"C:\Users\zeyua\Downloads\training_data"
DATA_YAML = os.path.join(DATASET_ROOT, "data.yaml")
PROJECT_DIR = os.path.join(os.path.dirname(os.getcwd()), "models", "mixed")
CHECKPOINT_NAME = "model_v2"

# Check if the dataset and yaml file exist
if not os.path.isdir(DATASET_ROOT):
    logging.error(f"Dataset directory not found at: {DATASET_ROOT}")
if not os.path.isfile(DATA_YAML):
    logging.error(f"data.yaml not found at: {DATA_YAML}")

# Load YOLOv11x model
try:
    model = YOLO("yolo11x.pt")
    logging.info("Successfully loaded yolov11x.pt")
except Exception as e:
    logging.error(f"Error loading model: {e}")
    logging.info("Please ensure 'yolov11x.pt' is in the correct directory or specify the correct path.")

logging.info("initialize variable successful")

INFO:root:Successfully loaded yolov11x.pt
INFO:root:initialize variable successful


# Dataset Diagnostics

Run this cell to check for common dataset issues that cause CUDA errors:

In [ ]:
# Run dataset diagnostics
%run diagnose_dataset.py

In [ ]:
# Clear CUDA cache and reset environment
import gc
import torch

# Clear Python garbage
gc.collect()

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"✓ CUDA cache cleared")
    print(f"✓ GPU memory allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"✓ GPU memory reserved: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")
else:
    print("⚠ CUDA not available")

print("✓ Environment reset complete")

In [3]:
# Train the model (optimized for cleaner relabeled dataset)

logging.info("Start training the model")
try:
    results = model.train(
                data=DATA_YAML,
                epochs=200,
                imgsz=640,
                batch=8, 
                
                # Core Optimization
                freeze=10,
                patience=30,
                device=0,
                workers=8,
                
                # Project Management
                project=PROJECT_DIR,
                name=CHECKPOINT_NAME,
                exist_ok=True,
                
                # Training Strategy
                pretrained=True,
                optimizer="AdamW",
                momentum=0.937,
                lr0=0.002,  # Slightly higher lr for cleaner dataset
                lrf=0.001,
                weight_decay=0.0005,
                
                # Data Augmentation
                cache=False,            
                augment=True,
                degrees=20.0,
                translate=0.15,
                scale=0.8,
                shear=3.0,
                perspective=0.0001,
                flipud=0.0,
                fliplr=0.5,
                mosaic=1.0,
                mixup=0.2,
                copy_paste=0.15,
                close_mosaic=15,
                
                # Advanced Techniques
                amp=True,
                rect=False,
                cos_lr=True,
                dropout=0.15,
                
                # Save Strategy
                save=True,
                save_period=10,
                
                # Monitoring
                plots=True,
                verbose=True,
                val=True
            )
    logging.info("Training completed")
except Exception as e:
    logging.error(f"An error occurred during training: {e}")
    logging.info("Please check your dataset path, CUDA setup, and for out-of-memory errors.")


INFO:root:Start training the model


New https://pypi.org/project/ultralytics/8.3.247 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.226  Python-3.14.0 torch-2.9.0+cu126 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 22528MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\zeyua\Downloads\training_data\data.yaml, degrees=20.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.15, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo11x.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=mode

INFO:root:Training completed
